# ViT Sprint 3 — Fine-tune Funnel (Colab A100)

Runs the **five** locked Sprint 3 fold-0 fine-tune candidates from
`007-vit-finetune.md` via `scripts/train.py`. Colab is a GPU runner only — all
model / training / metric logic lives in the repository.

**Gates (VLD-11):** training is the only A100-eligible stage. The env cell
hard-asserts an A100; `scripts/train.py` also aborts off-A100 internally. Do
**not** pass `--allow-non-a100` for real runs, and do not recompute frozen
feature caches here.

**Candidates (fold 0 only, VLD-12):**
`02_single_swin_t`, `04_pair_vit_b_swin_t_concat`, `05_pair_vit_b_beit_b_concat`,
`09_pair_swin_t_beit_b_weighted`, `11_triple_weighted` (all `_finetune_official`).

Prerequisite: the `sprint3/vit-finetune-funnel` branch (Slices 1–3) must be
pushed to GitHub before running cell 2.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the Sprint 3 branch and record the commit
import os, subprocess
REPO_URL = 'https://github.com/YasinEkici/hyperkvasir-multi-backbone-fusion.git'
BRANCH = 'sprint3/vit-finetune-funnel'
REPO_DIR = '/content/hyperkvasir-multi-backbone-fusion'
# Public repo: no token needed. For a private repo, set
# os.environ['GITHUB_TOKEN'] = '...' in a scratch cell BEFORE running this one.
token = os.environ.get('GITHUB_TOKEN', '')
clone_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@') if token else REPO_URL
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only'], check=True)
elif os.path.exists(REPO_DIR):
    raise RuntimeError(f'Path exists but is not a git repo: {REPO_DIR}. Restart the runtime.')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', clone_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git rev-parse HEAD

In [ ]:
# 3. Build the isolated Colab env and hard-assert an A100 (VLD-11)
import os
REPO_DIR = '/content/hyperkvasir-multi-backbone-fusion'
os.chdir(REPO_DIR)
required_files = [
    'pyproject.toml',
    'env/requirements-colab.txt',
    'configs/vit/training/vit_finetune.yaml',
    'configs/vit/experiment_matrix.yaml',
]
missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    !git branch --show-current
    !git rev-parse HEAD
    raise FileNotFoundError(f'Missing required repo files: {missing}. Push Slices 1-3 to GitHub and rerun cell 2.')
!python -m pip install -q uv
!uv venv --python 3.11 .venv
# Use `uv pip install -r` (resolves transitive deps); never `uv sync` here
# (local pyproject is pinned to CUDA 13.2 for the RTX 5080).
!uv pip install --python .venv/bin/python -r env/requirements-colab.txt
!uv run --no-sync python -c "import torch, timm; print('torch', torch.__version__, 'timm', timm.__version__); assert torch.cuda.is_available(), 'CUDA unavailable'; name=torch.cuda.get_device_name(0); print('device', name); assert 'A100' in name, f'A100 required, found {name}'; print(torch.ones(1, device='cuda'))"

In [ ]:
# 4. CONTROL PANEL — the five locked candidates + Drive root. Edit only here.
import os
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive/hyperkvasir-multi-cnn-fusion'
EXPERIMENTS = [
    '02_single_swin_t_finetune_official',          # smoke first (cheapest)
    '04_pair_vit_b_swin_t_concat_finetune_official',
    '05_pair_vit_b_beit_b_concat_finetune_official',
    '09_pair_swin_t_beit_b_weighted_finetune_official',
    '11_triple_weighted_finetune_official',         # heaviest (triple)
]
print('Drive root:', os.environ['DRIVE_ROOT'])
print('Experiments:', *EXPERIMENTS, sep='\n  ')

In [ ]:
# 5. Stage the approved Drive dataset to the path the manifest expects
#    (project-relative data/raw/hyperkvasir/labeled-images). Fine-tuning reads
#    images, NOT feature caches — caches are not needed on Colab (VLD-11).
import os, shutil
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
src = f"{os.environ['DRIVE_ROOT']}/data/hyperkvasir/labeled-images"
if not os.path.isdir(src):
    raise FileNotFoundError(f'Approved dataset not found on Drive: {src}')
dst = 'data/raw/hyperkvasir/labeled-images'
os.makedirs('data/raw/hyperkvasir', exist_ok=True)
if os.path.isdir(dst):
    print(f'[skip] already staged at {dst}')
else:
    shutil.copytree(src, dst)
n_files = sum(len(files) for _, _, files in os.walk(dst))
print(f'[OK] staged {n_files} files -> {dst}')

In [ ]:
# 6. Dataset + git provenance gate (reuses the CNN D-09 gate; VLD-11)
import os, subprocess
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
os.environ['EXPECTED_GIT_SHA'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
!uv run --no-sync python scripts/check_provenance.py --run-id sprint3_vit_finetune --source-dataset-root "$DRIVE_ROOT/data/hyperkvasir/labeled-images" --staged-dataset-root data/raw/hyperkvasir/labeled-images --manifest data/splits/hyperkvasir_official_5fold/fold_0.csv --approved-source "$DRIVE_ROOT/data/hyperkvasir/labeled-images" --expected-git-sha $EXPECTED_GIT_SHA --device cuda --output-root results/vit/runs

In [ ]:
# 7. Train the five fine-tune rows on fold 0. Resumable: skips any run that
#    already has metrics.json (re-run this cell after a session drop).
import os, subprocess
from pathlib import Path
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
runs = Path('results/vit/runs')
for exp in EXPERIMENTS:
    if (runs / exp / 'metrics.json').exists():
        print(f'[skip] {exp} already complete'); continue
    print(f'\n===== [run] {exp} =====')
    subprocess.run([
        'uv', 'run', '--no-sync', 'python', 'scripts/train.py',
        '--config', 'configs/vit/experiment_matrix.yaml',
        '--experiment', exp, '--device', 'cuda',
    ], check=True)
print('\n[done] all requested fine-tune runs finished')

In [ ]:
# 8. Artifact + finite-metric checklist (hard-fail on any problem)
import json
from pathlib import Path
import math
runs = Path('results/vit/runs')
required = ['metrics.json', 'config.yaml', 'predictions.npz', 'best.pt']
problems = []
for exp in EXPERIMENTS:
    present = {r: (runs / exp / r).exists() for r in required}
    line = {k: v for k, v in present.items()}
    if all(present.values()):
        m = json.load((runs / exp / 'metrics.json').open())['test']
        finite = all(isinstance(m.get(k), (int, float)) and math.isfinite(m[k])
                     for k in ('macro_f1', 'accuracy', 'macro_precision', 'macro_recall'))
        print(f"{exp}: f1={m['macro_f1']:.4f} acc={m['accuracy']:.4f} finite={finite}")
        if not finite:
            problems.append(f'{exp}: non-finite test metric')
    else:
        print(f'{exp}: MISSING {line}')
        problems.append(f'{exp}: missing artifacts {line}')
if problems:
    raise RuntimeError('Artifact/metric problems:\n  ' + '\n  '.join(problems))
print('\n[OK] all five runs have the 4 artifacts and finite test metrics')

In [ ]:
# 9. Copy run dirs back to Drive (persists) and zip for direct download.
import os, shutil
from pathlib import Path
os.chdir('/content/hyperkvasir-multi-backbone-fusion')
dst_root = Path(os.environ['DRIVE_ROOT']) / 'returned_outputs' / 'sprint3_vit_finetune'
dst_root.mkdir(parents=True, exist_ok=True)
for exp in EXPERIMENTS:
    src = Path('results/vit/runs') / exp
    if src.exists():
        shutil.copytree(src, dst_root / exp, dirs_exist_ok=True)
zip_path = shutil.make_archive('/content/vit_finetune_runs', 'zip', 'results/vit/runs')
print('[OK] Drive copy:', dst_root)
print('[OK] zip:', zip_path)

In [ ]:
# 10. Download the zip to your machine (unzip into local results/vit/runs/).
from google.colab import files
files.download('/content/vit_finetune_runs.zip')